# 04. 이상 탐지 모델링

3가지 비지도 학습 모델을 적용하여 선박 이상 행동을 탐지한다.

| 모델 | 원리 | 장점 |
|------|------|------|
| Isolation Forest | 랜덤 분할로 이상치 격리 | 고차원, 대용량에 강함 |
| DBSCAN | 밀도 기반 클러스터링 | 임의 형태 클러스터 탐지 |
| LOF | 지역 밀도 비교 | 국소적 이상치에 강함 |

최종: 3개 모델 앙상블 (2개 이상 동의 시 이상 판정)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.models import (
    detect_isolation_forest, detect_lof, detect_dbscan,
    ensemble_anomaly, FEATURE_COLS
)

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_parquet('../data/ais_featured.parquet')
print(f'{len(df):,} rows, {df["MMSI"].nunique()} vessels')

# 결측치 제거 (피처 컬럼 기준)
df_model = df.dropna(subset=FEATURE_COLS).copy()
print(f'모델 입력: {len(df_model):,} rows (결측 제거 후)')

## 4.1 Isolation Forest

In [ ]:
df_model = detect_isolation_forest(df_model, contamination=0.05)

n_anomaly = df_model['anomaly_if'].sum()
print(f'Isolation Forest 이상 탐지: {n_anomaly:,} ({n_anomaly/len(df_model)*100:.2f}%)')

## 4.2 Local Outlier Factor

In [ ]:
df_model = detect_lof(df_model, contamination=0.05)

n_anomaly = df_model['anomaly_lof'].sum()
print(f'LOF 이상 탐지: {n_anomaly:,} ({n_anomaly/len(df_model)*100:.2f}%)')

## 4.3 DBSCAN

In [ ]:
df_model = detect_dbscan(df_model, eps=1.5, min_samples=20)

n_anomaly = df_model['anomaly_dbscan'].sum()
print(f'DBSCAN 이상 탐지: {n_anomaly:,} ({n_anomaly/len(df_model)*100:.2f}%)')

## 4.4 모델 간 비교

In [ ]:
anomaly_cols = ['anomaly_if', 'anomaly_lof', 'anomaly_dbscan']
model_names = ['Isolation Forest', 'LOF', 'DBSCAN']

# 모델별 이상 탐지 수
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = [df_model[c].sum() for c in anomaly_cols]
axes[0].bar(model_names, counts, color=['steelblue', 'coral', 'green'], alpha=0.8)
axes[0].set_ylabel('Anomaly Count')
axes[0].set_title('Anomalies Detected by Each Model')
for i, v in enumerate(counts):
    axes[0].text(i, v + 100, f'{v:,}', ha='center')

# 모델 간 합의
agreement = df_model[anomaly_cols].sum(axis=1).value_counts().sort_index()
axes[1].bar(agreement.index, agreement.values, color='purple', alpha=0.7)
axes[1].set_xlabel('Number of Models Agreeing (Anomaly)')
axes[1].set_ylabel('Record Count')
axes[1].set_title('Model Agreement Distribution')
axes[1].set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.savefig('../results/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 앙상블 (최종 판정)

In [ ]:
df_model = ensemble_anomaly(df_model, threshold=2)

n_final = df_model['anomaly_final'].sum()
n_vessels = df_model[df_model['anomaly_final'] == 1]['MMSI'].nunique()

print(f'=== 최종 앙상블 결과 ===')
print(f'이상 레코드: {n_final:,} / {len(df_model):,} ({n_final/len(df_model)*100:.2f}%)')
print(f'이상 선박: {n_vessels} / {df_model["MMSI"].nunique()}')

## 4.6 이상 탐지 결과 분석

In [ ]:
# 정상 vs 이상 피처 비교
normal = df_model[df_model['anomaly_final'] == 0]
anomaly = df_model[df_model['anomaly_final'] == 1]

print('=== 정상 vs 이상 피처 평균 비교 ===')
comparison = pd.DataFrame({
    'Normal': normal[FEATURE_COLS].mean(),
    'Anomaly': anomaly[FEATURE_COLS].mean(),
    'Ratio': anomaly[FEATURE_COLS].mean() / normal[FEATURE_COLS].mean().replace(0, 1)
})
print(comparison.round(2))

In [ ]:
# 피처별 정상 vs 이상 분포 비교
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, col in enumerate(['SOG', 'speed_deviation', 'course_change', 'signal_gap_sec']):
    ax = axes[idx // 2, idx % 2]
    clip_val = normal[col].quantile(0.99)
    normal[col].clip(upper=clip_val).hist(bins=40, ax=ax, alpha=0.5, color='blue', label='Normal', density=True)
    anomaly[col].clip(upper=clip_val).hist(bins=40, ax=ax, alpha=0.5, color='red', label='Anomaly', density=True)
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/normal_vs_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 결과 저장

In [ ]:
df_model.to_parquet('../data/ais_results.parquet', index=False)

import os
fsize = os.path.getsize('../data/ais_results.parquet') / 1e6
print(f'저장 완료: data/ais_results.parquet ({fsize:.1f} MB)')